# Code Generation with GraphRAG

## Introduction

In this notebook we show how to GraphRAG can be used to traverse through python library documentation in order to provide help to an LLM that is generating code.

We use a custom Strategy to traverse through the documents and specifically select nodes that contain code examples and/or descriptive text.

## Environment Setup

The following block will configure the environment from the Colab Secrets.
To run it, you should have the following Colab Secrets defined and accessible to this notebook:

- `OPENAI_API_KEY`: The OpenAI key.
- `ASTRA_DB_API_ENDPOINT`: The Astra DB API endpoint.
- `ASTRA_DB_APPLICATION_TOKEN`: The Astra DB Application token.
- `LANGCHAIN_API_KEY`: Optional. If defined, will enable LangSmith tracing.
- `ASTRA_DB_KEYSPACE`: Optional. If defined, will specify the Astra DB keyspace. If not defined, will use the default.

In [ ]:
# Install modules.
#
# On Apple hardware, "spacy[apple]" will improve performance.
%pip install \
    langchain-core \
    langchain-astradb \
    langchain-openai \
    langchain-graph-retriever \
    spacy \
    graph-rag-example-helpers

The last package -- `graph-rag-example-helpers` -- includes some helpers for setting up the environment and provides the documents we will use for this example.

In [7]:
# Configure import paths.
import os
import sys
from pprint import pprint

sys.path.append("../../")

# Initialize environment variables.
from graph_rag_example_helpers.env import Environment, initialize_environment

initialize_environment(Environment.ASTRAPY)

os.environ["LANGCHAIN_PROJECT"] = "code-generation"

## Part 1: Loading Data

First, we'll demonstrate how to load the example AstraPy documentation into `AstraDBVectorStore`. We will be creating a document for every module, class, attribute, and function in the package. 

Note that we will use the pydoc description field for the page_content field in the document. Because of this, we will have many documents that have no page content. We will store the rest of the item details in the document metadata.

#### Example doc with page content

<details markdown><summary>Click to expand</summary>

```yaml
id: astrapy.client.DataAPIClient

page_content: |
    A client for using the Data API. This is the main entry point and sits
    at the top of the conceptual "client -> database -> collection" hierarchy.

    A client is created first, optionally passing it a suitable Access Token.
    Starting from the client, then:
        - databases (Database and AsyncDatabase) are created for working with data
        - AstraDBAdmin objects can be created for admin-level work

metadata:
    name: DataAPIClient
    kind: class
    path: astrapy.client.DataAPIClient
    parameters: 
        token: |
            str | TokenProvider | None = None
            an Access Token to the database. Example: `"AstraCS:xyz..."`.
            This can be either a literal token string or a subclass of
            `astrapy.authentication.TokenProvider`.
        
        environment: |
            str | None = None
            a string representing the target Data API environment.
            It can be left unspecified for the default value of `Environment.PROD`;
            other values include `Environment.OTHER`, `Environment.DSE`.
        
        callers: |
            Sequence[CallerType] = []
            a list of caller identities, i.e. applications, or frameworks,
            on behalf of which Data API and DevOps API calls are performed.
            These end up in the request user-agent.
            Each caller identity is a ("caller_name", "caller_version") pair.

    example: |
        >>> from astrapy import DataAPIClient
        >>> my_client = DataAPIClient("AstraCS:...")
        >>> my_db0 = my_client.get_database(
        ...     "https://01234567-....apps.astra.datastax.com"
        ... )
        >>> my_coll = my_db0.create_collection("movies", dimension=2)
        >>> my_coll.insert_one({"title": "The Title", "$vector": [0.1, 0.3]})
        >>> my_db1 = my_client.get_database("01234567-...")
        >>> my_db2 = my_client.get_database("01234567-...", region="us-east1")
        >>> my_adm0 = my_client.get_admin()
        >>> my_adm1 = my_client.get_admin(token=more_powerful_token_override)
        >>> database_list = my_adm0.list_databases()

    references: 
        astrapy.client.DataAPIClient

    gathered_types: 
        astrapy.constants.CallerType
        astrapy.authentication.TokenProvider
```
</details>

This is the documentation for `astrapy.client.DataAPIClient`. The `page_content` field contains the description of the class, and the `metadata` field contains the rest of the details.

The `references` metadata field contains the list of related items used in the example code block. The `gathered_types` field contains the list of types from the parameters section. In GraphRAG, we can use these fields to link to other documents.

#### Example doc without page content

<details markdown><summary>Click to expand</summary>

```yaml
id: astrapy.admin.AstraDBAdmin.callers

page_content: ""

metadata:
  name: callers
  path: astrapy.admin.AstraDBAdmin.callers
  kind: attribute
```

</details>

This is the documentation for `astrapy.admin.AstraDBAdmin.callers`. The `page_content` field is empty, and the `metadata` field contains the details.

Despite having no page content, this document can still be useful.  We'll add a `parent` field to the metadata at vector store insertion time to link it to the parent document: `astrapy.admin.AstraDBAdmin`, and we can use this to traverse the graph.

### Create the AstraDBVectorStore
Next, we create the Vector Store we're going to load these documents into.
In our case, we use DataStax Astra DB with Open AI embeddings.

In [2]:
from langchain_astradb import AstraDBVectorStore
from langchain_openai import OpenAIEmbeddings

store = AstraDBVectorStore(
    embedding=OpenAIEmbeddings(),
    collection_name="code_generation"
)

### Loading Data into the Store
Next, we perform the actual loading. There are only about 1000 documents, so this should be quick.

We will use the ParentTransformer to add a parent field to the metadata document pages of kind. This will allow us to traverse the graph from a child to its parent.

In [ ]:
from graph_rag_example_helpers.datasets.astrapy import fetch_documents
from langchain_graph_retriever.transformers import ParentTransformer

transformer = ParentTransformer(path_delimiter=".")
doc_ids = store.add_documents(transformer.transform_documents(fetch_documents()))

We can retrieve a sample document to check if the parent field was added correctly:

In [10]:
pprint(store.get_by_document_id("astrapy.admin.AstraDBAdmin.callers").model_dump())

{'id': 'astrapy.admin.AstraDBAdmin.callers',
 'metadata': {'kind': 'attribute',
              'name': 'callers',
              'parent': 'astrapy.admin.AstraDBAdmin',
              'path': 'astrapy.admin.AstraDBAdmin.callers',
              'value': 'callers = callers_param'},
 'page_content': '',
 'type': 'Document'}


At this point, we've created a `VectorStore` with all the documents from the AstraPy documentation. Each document contains metadata about the module, class, attribute, or function, and the page content contains the description of the item.

In the next section we'll see how to build relationships from the metadata in order to traverse through the documentation in a similar way to how a human would.

## Part 2: Graph Traversal

The GraphRAG library allows us to traverse through the documents in the Vector Store.  By changing the Strategy, we can control how the traversal is performed.

### Basic Traversal

We'll start with the default `Eager` strategy, which will traverse the graph in a breadth-first manner. In order to do this we need to set up the relationships between the documents. This is done by defining the "edges" between the documents.

In our case we will connect the "references", "gathered_types", "parent", "implemented_by", and "bases" fields in the metadata to the "id" field of the document they reference.

In [11]:
edges = [
    ("gathered_types", "$id"),
    ("references", "$id"),
    ("parent", "$id"),
    ("implemented_by", "$id"),
    ("bases", "$id"),
]

Note that edges are directional, and indicate metadata fields by default.  The magic string `$id` is used to indicate the document's id.

In the above `edges` list, any document id found in `gathered_types` will be connected to documents with the corresponding id.

In [12]:
from langchain_graph_retriever import GraphRetriever

default_retriever = GraphRetriever(store=store, edges=edges)

default_retriever.invoke("Can you show me an example of initiating an astrapy client with token authentication? Also show me how to retrieve some rows from the database.")

[Document(id='astrapy.client.DataAPIClient', metadata={'_depth': 0, '_similarity_score': 0.7603516066015821, 'kind': 'class', 'name': 'DataAPIClient', 'path': 'astrapy.client.DataAPIClient', 'parameters': [{'name': 'token', 'type': 'str | TokenProvider | None', 'description': 'an Access Token to the database. Example: `"AstraCS:xyz..."`.\nThis can be either a literal token string or a subclass of\n`astrapy.authentication.TokenProvider`.', 'value': 'None', 'default': 'None'}, {'name': 'environment', 'type': 'str | None', 'description': 'a string representing the target Data API environment.\nIt can be left unspecified for the default value of `Environment.PROD`;\nother values include `Environment.OTHER`, `Environment.DSE`.', 'value': 'None', 'default': 'None'}, {'name': 'callers', 'type': 'Sequence[CallerType]', 'description': 'a list of caller identities, i.e. applications, or frameworks,\non behalf of which Data API and DevOps API calls are performed.\nThese end up in the request user

### Custom Strategy

Now we will create a custom strategy that will traverse a larger portion of the graph and return the documents that contain code examples or descriptive text. 

In [ ]:
import dataclasses
from graph_retriever.strategies import NodeTracker, Strategy
from graph_retriever.types import Node
from collections.abc import Iterable

@dataclasses.dataclass
class CodeExamples(Strategy):

    # internal dictionary to store all nodes found during the traversal
    _nodes: dict[str, Node] = dataclasses.field(default_factory=dict)

    def iteration(self, *, nodes: Iterable[Node], tracker: NodeTracker) -> None:
        # save all newly found nodes to the internal node dictionary for later use
        self._nodes.update({n.id: n for n in nodes})
        # traverse the newly found nodes
        new_count = tracker.traverse(nodes=nodes)

        # if no new nodes were found, we have reached the end of the traversal
        if new_count == 0:
            example_nodes = []
            description_nodes = []

            # iterate over all nodes and separate nodes with examples from nodes with
            # descriptions
            for node in self._nodes.values():
                if "example" in node.metadata:
                    example_nodes.append(node)
                elif node.content != "":
                    description_nodes.append(node)

            # select the nodes with examples first and descriptions second
            # note: the base `finalize_nodes` method will truncate the list to the
            #   `select_k` number of nodes
            tracker.select(example_nodes)
            tracker.select(description_nodes)

In [ ]:
graph_retriever = GraphRetriever(
    store=store,
    edges=edges,
    strategy=CodeExamples(max_depth=2)
)